# Pneumonia Classification - Pediatric Chest X-Ray Dataset
## Training a Binary Classifier (Transfer Learning)

This notebook loads the `chest_xray` dataset (which is already organized into `train`, `test`, and `val` folders with `NORMAL` and `PNEUMONIA` subdirectories) and trains an AI to classify them.


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 1. Load the Dataset\nUsing `image_dataset_from_directory` makes loading pre-segregated folders incredibly easy and memory-efficient.


In [ ]:
DATA_DIR = "C:/Users/prakh/Downloads/archive (1)/chest_xray"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")
TEST_DIR = os.path.join(DATA_DIR, "test")

BATCH_SIZE = 32
IMG_SIZE = (224, 224)

print("Loading Training set...")
train_dataset = image_dataset_from_directory(TRAIN_DIR, shuffle=True, batch_size=BATCH_SIZE, image_size=IMG_SIZE, label_mode='binary')

print("\nLoading Validation set...")
val_dataset = image_dataset_from_directory(VAL_DIR, shuffle=False, batch_size=BATCH_SIZE, image_size=IMG_SIZE, label_mode='binary')

print("\nLoading Test set...")
test_dataset = image_dataset_from_directory(TEST_DIR, shuffle=False, batch_size=BATCH_SIZE, image_size=IMG_SIZE, label_mode='binary')

class_names = train_dataset.class_names
print(f"\nClasses: {class_names}")

## 2. Visualize the Data


In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_dataset.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        
        # labels are shape (batch_size, 1), so we extract the scalar
        lbl_idx = int(labels[i].numpy()[0])
        plt.title(class_names[lbl_idx])
        plt.axis("off")
plt.tight_layout()
plt.show()

## 3. Data Augmentation & Preprocessing
Data augmentation (randomly flipping and zooming images) helps the model learn from limited data without overfitting.


In [ ]:
# Data augmentation pipeline
data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip('horizontal'),
  tf.keras.layers.RandomRotation(0.1),
  tf.keras.layers.RandomZoom(0.1),
])

# DenseNet requires inputs from -1 to 1 or specific ImageNet normalization. 
# We'll use the built-in preprocess_input function.
preprocess_input = tf.keras.applications.densenet.preprocess_input

## 4. Build the Model (Transfer Learning)
We will use a pretrained **DenseNet121** (trained on ImageNet) as the base, and add our own binary classification head on top.


In [ ]:
# Create the base model from the pre-trained model DenseNet121
base_model = DenseNet121(input_shape=(224, 224, 3), include_top=False, weights='imagenet')

# Freeze the base model (we don't want to destroy the pre-trained weights during initial training)
base_model.trainable = False

# Build the final model
inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.2)(x) # Dropout prevents overfitting
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs, outputs)

# Compile the model
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss=tf.keras.losses.BinaryCrossentropy(),
              metrics=['accuracy'])

model.summary()

## 5. Train the Model


In [ ]:
# Callbacks to stop early if the model stops improving, and save the best weights
callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True, monitor='val_loss'),
    ModelCheckpoint('best_pneumonia_model.keras', save_best_only=True, monitor='val_loss')
]

EPOCHS = 10

# Prefetching makes training faster
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.prefetch(buffer_size=AUTOTUNE)

print("Starting training (this might take a few minutes depending on your CPU/GPU)...")
history = model.fit(train_dataset,
                    epochs=EPOCHS,
                    validation_data=val_dataset,
                    callbacks=callbacks)

## 6. Evaluate Training History


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy', lw=2)
plt.plot(val_acc, label='Validation Accuracy', lw=2)
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss', lw=2)
plt.plot(val_loss, label='Validation Loss', lw=2)
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

## 7. Test Set Evaluation
Finally, let's see how our trained model performs on the unseen data from the `test` folder.


In [ ]:
print("Evaluating on Test Dataset...")
# Ensure test dataset is also prefetched
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

loss, accuracy = model.evaluate(test_dataset)
print(f'Test accuracy: {accuracy:.4f}')
print(f'Test loss: {loss:.4f}')

## 8. Detailed Metrics
Let's look at the Confusion Matrix to see exactly where the model gets confused.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

gnd_truths = []
predictions = []

# Iterate through test dataset to collect predictions
for images, labels in test_dataset:
    preds = model.predict(images, verbose=0)
    predictions.extend((preds > 0.5).astype(int).flatten())
    gnd_truths.extend(labels.numpy().flatten())

print("Classification Report:")
print(classification_report(gnd_truths, predictions, target_names=class_names))

cm = confusion_matrix(gnd_truths, predictions)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()